# Phase 3 — QLoRA Fine-Tuning
### Train Qwen 1.5B on Spider SQL Dataset

This notebook:
1. Loads the formatted Spider dataset
2. Configures QLoRA (4-bit quantization + LoRA adapters)
3. Trains using SFTTrainer with W&B logging
4. Saves the LoRA adapters to disk


In [ ]:
import os
import torch
import wandb
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print("All imports successful!")
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name      : {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Total VRAM    : {vram:.1f} GB")


## Step 1 — Training Configuration

In [ ]:
# ── All hyperparameters in one place ─────────────────────────────
CONFIG = {
    # Model
    "model_name"        : "Qwen/Qwen1.5-1.8B-Chat",
    "data_dir"          : "./data/spider_formatted",

    # LoRA hyperparameters
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "lora_target"       : ["q_proj", "v_proj", "k_proj", "o_proj"],

    # Training hyperparameters
    "num_epochs"        : 5,       # ← changed
    "batch_size"        : 8,       # ← changed
    "grad_accum_steps"  : 2,       # ← changed
    "learning_rate"     : 2e-4,
    "max_seq_length"    : 512,
    "warmup_ratio"      : 0.05,
    "lr_scheduler"      : "cosine",
    "weight_decay"      : 0.01,

    # Output
    "output_dir"        : "./outputs/qwen-sql-qlora",
    "save_steps"        : 100,
    "eval_steps"        : 100,
    "logging_steps"     : 10,

    # W&B
    "wandb_project"     : "sql-finetuning",
    "wandb_run_name"    : "qwen1.5-spider-qlora-r16-v2",  # ← v2 for new run
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("Configuration ready!")
print(f"\nKey hyperparameters:")
print(f"  LoRA rank       : {CONFIG['lora_r']}")
print(f"  LoRA alpha      : {CONFIG['lora_alpha']}")
print(f"  Learning rate   : {CONFIG['learning_rate']}")
print(f"  Epochs          : {CONFIG['num_epochs']}")
print(f"  Batch size      : {CONFIG['batch_size']} x {CONFIG['grad_accum_steps']} = {CONFIG['batch_size'] * CONFIG['grad_accum_steps']} effective")


## Step 2 — Initialize Weights & Biases

In [ ]:
# Initialize W&B run
wandb.init(
    project = CONFIG["wandb_project"],
    name    = CONFIG["wandb_run_name"],
    config  = CONFIG,
)

print(f"W&B run initialized!")
print(f"Project  : {CONFIG['wandb_project']}")
print(f"Run name : {CONFIG['wandb_run_name']}")
print(f"Track at : https://wandb.ai")


## Step 3 — Load Dataset

In [ ]:
dataset = load_from_disk(CONFIG["data_dir"])

# 500 samples for fast training
train_data = dataset["train"].select(range(3500))
val_data   = dataset["validation"].select(range(200))

print(f"Train samples      : {len(train_data)}")
print(f"Validation samples : {len(val_data)}")

## Step 4 — Configure 4-bit Quantization (QLoRA)

In [ ]:
# BitsAndBytes 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,          # load model in 4-bit
    bnb_4bit_quant_type       = "nf4",         # NormalFloat4 quantization
    bnb_4bit_compute_dtype    = torch.float16, # compute in fp16
    bnb_4bit_use_double_quant = True,          # nested quantization
)

print("4-bit quantization config ready!")
print("  Type             : NF4 (NormalFloat4)")
print("  Compute dtype    : float16")
print("  Double quant     : True")
print("\nThis reduces model VRAM from ~7GB to ~1.5GB!")


## Step 5 — Load Model with Quantization

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"
print("Tokenizer loaded!")

# Load model in 4-bit
print("\nLoading model in 4-bit (from cache, should be fast)...")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config = bnb_config,
    device_map          = {"": 0},    # ← changed from "cuda" to {"": 0}
    trust_remote_code   = True,
)
model.config.use_cache = False

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nModel loaded in 4-bit!")
print(f"VRAM used : {vram_used:.2f} GB (vs ~7GB in fp16)")

## Step 6 — Apply LoRA Adapters

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r              = CONFIG["lora_r"],
    lora_alpha     = CONFIG["lora_alpha"],
    lora_dropout   = CONFIG["lora_dropout"],
    target_modules = CONFIG["lora_target"],
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nVRAM after LoRA : {vram_used:.2f} GB")
print("Only the small LoRA adapters will be trained!")


## Step 7 — Training Arguments

In [ ]:
training_args = SFTConfig(
    # Output
    output_dir                  = CONFIG["output_dir"],

    # Training schedule
    num_train_epochs            = CONFIG["num_epochs"],
    per_device_train_batch_size = CONFIG["batch_size"],
    per_device_eval_batch_size  = CONFIG["batch_size"],
    gradient_accumulation_steps = CONFIG["grad_accum_steps"],

    # Optimizer
    learning_rate               = CONFIG["learning_rate"],
    weight_decay                = CONFIG["weight_decay"],
    lr_scheduler_type           = CONFIG["lr_scheduler"],
    warmup_ratio                = CONFIG["warmup_ratio"],

    # Precision
    fp16                        = True,
    bf16                        = False,

    # Logging & saving
    logging_steps               = CONFIG["logging_steps"],
    save_steps                  = CONFIG["save_steps"],
    eval_steps                  = CONFIG["eval_steps"],
    evaluation_strategy         = "steps",
    save_strategy               = "steps",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",

    # W&B
    report_to                   = "wandb",
    run_name                    = CONFIG["wandb_run_name"],

    # Sequence length
    max_seq_length              = CONFIG["max_seq_length"],
    dataset_text_field          = "text",

    # Misc
    dataloader_pin_memory       = False,
    group_by_length             = True,
)

print("Training arguments configured!")


## Step 8 — Initialize Trainer

In [ ]:
trainer = SFTTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_data,
    eval_dataset    = val_data,
    tokenizer       = tokenizer,
)

print("Trainer initialized!")
print(f"\nTraining summary:")
print(f"  Total samples     : {len(train_data)}")
print(f"  Epochs            : {CONFIG['num_epochs']}")
print(f"  Effective batch   : {CONFIG['batch_size'] * CONFIG['grad_accum_steps']}")
total_steps = (len(train_data) // (CONFIG['batch_size'] * CONFIG['grad_accum_steps'])) * CONFIG['num_epochs']
print(f"  Total steps       : ~{total_steps}")
print(f"  Saving to         : {CONFIG['output_dir']}")
print(f"\nEstimated training time on RTX 5060: ~45-90 minutes")


## Step 9 — Start Training! 🔥

In [ ]:
print("Starting training...")
print("Watch your W&B dashboard for live loss curves!")
print("=" * 50)

# Train!
trainer.train()

print("=" * 50)
print("Training complete!") 


## Step 10 — Save LoRA Adapters

In [ ]:
# Save the LoRA adapters (NOT the full model - much smaller!)
adapter_path = CONFIG["output_dir"] + "/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapters saved to: {adapter_path}")

# Check adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter size: {total_size / 1024**2:.1f} MB (vs ~3.5GB for full model!)")

# Finish W&B run
wandb.finish()
print("\nW&B run finished!")
print("\nPhase 3 Complete!")
print("LoRA adapters saved and ready for evaluation.")
print("Next up: Phase 4 - Post Fine-Tuning Evaluation!")
